# Aşama 3: Matematiksel Model ve Batarya Simülatörü (Ön Doğrulama)

Bu çalışma defteri `src/battery_model.py` içerisindeki 1-RC Thevenin + birleştirilmiş termal + ampirik yaşlanma ODE sistemini ve sayısal çözücüleri (Euler, RK4, `solve_ivp`) analitik çözümlerle ve gerçek Stanford `W8` C/20 OCV eğrisiyle doğrular.

> **Not:** Aşama 4 parametre kestirimi henüz yapılmadığı için bu defterdeki direnç ve termal değerler açıkça **kalibrasyon öncesi ön parametreler** olarak etiketlenmiştir.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.battery_model import (
    BatteryParameters,
    OCVInterpolator,
    euler_stability_limit,
    simulate_current_profile,
    simulate_soc_profile,
)

ocv = OCVInterpolator.from_processed_csv(PROJECT_ROOT / 'data' / 'processed' / 'stanford_ocv_c20_diag1.csv', cell_id='W8')
params = BatteryParameters()
print('Kalibre edildi mi?:', params.is_calibrated)
print('Açık Euler kararlılık adım sınırı [s]:', euler_stability_limit(params))

## 1. Analitik Doğrulama (Sabit Akım RC ve Sıfır Akım Termal Soğuma)

In [ ]:
t_rc = np.linspace(0.0, 180.0, 181)
res_rc = simulate_current_profile(t_rc, np.full_like(t_rc, 4.0), params, lambda s: 3.6 + 0.5*s, method='rk4')
tau_rc = params.R1_Ohm * params.C1_F
vp_exact = 4.0 * params.R1_Ohm * (1.0 - np.exp(-t_rc / tau_rc))
print(f'RK4 Vp maks. hata: {np.max(np.abs(res_rc["vp_V"] - vp_exact)):.2e} V')

## 2. Kalibrasyon Öncesi Ön Davranış Kontrolü (0.5C, 1C, 2C | SOC: 0.20 → 0.90)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for label, i_val in [('0.5C (2.425 A)', 2.425), ('1C (4.850 A)', 4.850), ('2C (9.700 A)', 9.700)]:
    sim = simulate_soc_profile([i_val]*7, params, ocv, soc_init=0.20, soc_target=0.90, method='rk4', dt_max_s=1.0)
    t_min = sim['time_s'] / 60.0
    axes[0, 0].plot(t_min, sim['soc'], linewidth=1.8, label=label)
    axes[0, 1].plot(t_min, sim['terminal_voltage_V'], linewidth=1.8, label=label)
    axes[1, 0].plot(t_min, sim['temperature_C'], linewidth=1.8, label=label)
    axes[1, 1].plot(t_min, sim['aging_D'], linewidth=1.8, label=label)
    print(f'{label}: tf={sim["final_time_s"]/60:.2f} dk, Vmax={sim["max_voltage_V"]:.3f} V, Tmax={sim["max_temperature_C"]:.2f} C, Eloss={sim["energy_loss_Wh"][-1]:.3f} Wh, D={sim["total_aging_D"]:.4f}')
axes[0, 0].set_title('SOC – Süre'); axes[0, 0].set_ylabel('SOC [-]')
axes[0, 1].set_title('Terminal Gerilimi – Süre'); axes[0, 1].set_ylabel('Gerilim [V]'); axes[0, 1].axhline(4.20, ls='--', c='gray')
axes[1, 0].set_title('Hücre Sıcaklığı – Süre'); axes[1, 0].set_ylabel('Sıcaklık [°C]'); axes[1, 0].set_xlabel('Süre [dk]')
axes[1, 1].set_title('Göreli Yaşlanma D(t) – Süre'); axes[1, 1].set_ylabel('Göreli Yaşlanma D [-]'); axes[1, 1].set_xlabel('Süre [dk]')
for ax in axes.ravel(): ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.suptitle('Kalibrasyon Öncesi Model Ön Davranış Kontrolü (W8 OCV + Ön Parametreler)', fontsize=13)
fig.tight_layout()
plt.show()

## 3. Aşama 5: Sayısal ODE Çözücüleri (Euler, RK4, solve_ivp) ve Sonlu Fark Gradyan Analizi

Kalibre edilmiş model (`calibrated_parameters.json`) üzerinde Euler, RK4 ve `solve_ivp` yöntemlerinin DOP853 referansına göre yakınsama mertebesi, adım boyu duyarlılığı, kararlılık sınırı ve sonlu fark gradyan hassasiyeti karşılaştırılır.

In [ ]:
from IPython.display import Image, display
from src.numerical_analysis import run_phase5_pipeline

res5 = run_phase5_pipeline(
    processed_dir=PROJECT_ROOT / 'data' / 'processed',
    tables_dir=PROJECT_ROOT / 'results' / 'tables',
    figures_dir=PROJECT_ROOT / 'results' / 'figures',
)
print('Ampirik yakınsama mertebeleri:', res5['convergence_orders'])
display(res5['ode_benchmark'])
display(res5['soc_dt_sensitivity'])
display(res5['fd_sensitivity'])
for fp in res5['saved_figures']:
    display(Image(filename=fp))